In [ ]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from func.UsefulFunctions import *

This notebook is for labelling the data from wikidata

In [ ]:
artistDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/Query 107 - Unlabelled People with musical occupations (+subclass of musical occupation) + gender + occupation.csv")
occupationsDF = pd.read_excel("/home/aja25/Downloads/ArtistsData_FromWikidata/Annotated musical occupations (and the occupations from the subclases).xlsx")
artists_with_countryDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/Query 109 – Unlabelled People with musical occupations (+subclass of musical occupation) + country of citizenship.csv")
artists_with_influencesDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/Query 110 - All Musical Occupationists and their influences (when they have them).csv")
artists_with_statementsDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/Query 29 - A selection of musical occupationists with the most statements.csv")
artists_with_genresDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/QUERY 108 - Unlabelled People with musical occupations (+subclass of musical occupation) + genre.csv")
artists_with_wikiDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/Query 20 - artists and their wikipedia pages.csv")
artists_with_musicBrainzDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/Query 24 - music occupationists and musicbrainz id.csv")
artists_with_recordLabelsDF = pd.read_csv("/home/aja25/Downloads/ArtistsData_FromWikidata/QUERY 106 - PERFROMERS from their Record labels + record label + gender.csv")
display(artistDF.head(5))
display(occupationsDF.head(5))
display(artists_with_countryDF.head(5))
display(artists_with_influencesDF.head(5))
display(artists_with_statementsDF.head(5))
display(artists_with_genresDF.head(5))
display(artists_with_wikiDF.head(5))
display(artists_with_musicBrainzDF.head(5))
display(artists_with_recordLabelsDF.head(5))


In [ ]:
artistDF = artistDF.map(getEntityIDValue)
occupationsDF.occupation = occupationsDF.occupation.map(getEntityIDValue)
artists_with_countryDF = artists_with_countryDF.map(getEntityIDValue)
artists_with_influencesDF[['artist','influencedBY']] = artists_with_influencesDF[['artist','influencedBY']].map(getEntityIDValue)
artists_with_statementsDF.artist = artists_with_statementsDF.artist.map(getEntityIDValue)
artists_with_genresDF = artists_with_genresDF.map(getEntityIDValue)
artists_with_musicBrainzDF.artist = artists_with_musicBrainzDF.artist.map(getEntityIDValue)
artists_with_recordLabelsDF[['artist','recordLabel','gender']] = artists_with_recordLabelsDF[['artist','recordLabel','gender']].map(getEntityIDValue)
artists_with_wikiDF.artist = artists_with_wikiDF.artist.map(getEntityIDValue)




display(artistDF.head(5))
display(occupationsDF.head(5))
display(artists_with_countryDF.head(5))
display(artists_with_influencesDF.head(5))
display(artists_with_statementsDF.head(5))
display(artists_with_genresDF.head(5))
display(artists_with_wikiDF.head(5))
display(artists_with_musicBrainzDF.head(5))
display(artists_with_recordLabelsDF.head(5))
display(artists_with_wikiDF.head(5))

In [ ]:
allowedOccupations = occupationsDF[occupationsDF["isItAllowed"] == "Yes"][["occupation","occupationLabel"]]
allowedOccupations.drop_duplicates(inplace=True)
display(allowedOccupations.head(5))

In [ ]:
#Inner merge is used so only the individuals with the allowed occupations are preserved
artistDF = artistDF.merge(allowedOccupations, left_on="occupation", right_on="occupation", how="inner")
display(artistDF.head(5))

In [ ]:
dict_gender = getLabels(artistDF.gender.dropna().unique().tolist())
artistDF["genderLabel"] = artistDF.gender.replace(dict_gender)
display(artistDF.head(5))

In [ ]:
#let's see the genders in the system
print(artistDF.genderLabel.unique())

In [ ]:
#Get all the individuals whose gender is something not recognised/not in the system
#
genderDiscards = artistDF[artistDF.gender.isin(artistDF.genderLabel)]
print(f"There are {len(genderDiscards.artist.unique())} individuals without no gender or an error")
display(genderDiscards.head(5))
genderDiscards.artist.drop_duplicates().to_csv("./missingValuesStore/artistsEntitiesWithoutGender.csv", index=False)

#Discard the unknown genders and the individuals
artistDF = artistDF[~artistDF.gender.isin(artistDF.genderLabel)]

In [ ]:
finalArtistDF = artistDF.merge(artists_with_countryDF, left_on="artist", right_on="artist", how="left")
finalArtistDF = finalArtistDF.merge(artists_with_influencesDF[['artist','influencedBY','influencedBYLabel']], left_on="artist", right_on="artist", how="left")
finalArtistDF = finalArtistDF.merge(artists_with_statementsDF,left_on="artist", right_on="artist", how="left")
finalArtistDF = finalArtistDF.merge(artists_with_genresDF, left_on="artist", right_on="artist", how="left")
finalArtistDF = finalArtistDF.merge(artists_with_wikiDF[['artist','article']], left_on="artist", right_on="artist", how="left")
finalArtistDF = finalArtistDF.merge(artists_with_musicBrainzDF, left_on="artist", right_on="artist", how="left")
finalArtistDF = finalArtistDF.merge(artists_with_recordLabelsDF[['artist','recordLabel','recordLabelLabel']], left_on="artist", right_on="artist", how="left")
display(finalArtistDF.head(5))

In [ ]:
print(list(finalArtistDF.columns))

dict_country = getLabels(finalArtistDF['countryCitizenship'].dropna().unique().tolist())
#make dictionary into a dataframe. This idea was gotten from a stackoverflow answer by Andy Hayden accessed 24/07/2026 on https://stackoverflow.com/a/18837389

country_pdDF = pd.DataFrame(dict_country.items(), columns=['countryCitizenship','countryCitizenshipLabel'])
finalArtistDF = finalArtistDF.merge(country_pdDF, left_on="countryCitizenship", right_on="countryCitizenship", how="left")

dict_genre = getLabels(finalArtistDF['genre'].dropna().unique().tolist())
genre_pdDF = pd.DataFrame(dict_genre.items(), columns=['genre','genreLabel'])
finalArtistDF = finalArtistDF.merge(genre_pdDF, left_on="genre", right_on="genre", how="left")
dict_artistName = getLabels(finalArtistDF['artist'].dropna().unique().tolist())
name_pdDF = pd.DataFrame(dict_artistName.items(), columns=['artist','artistLabel'])
finalArtistDF = finalArtistDF.merge(name_pdDF, left_on="artist", right_on="artist", how="left")

display(finalArtistDF.head(5))



In [ ]:
#Save_file
finalArtistDF.to_parquet("pipelinesArtistsWikidata.parquet",index=False)